# Camp de Blume-Capel vs transicions reals — inferir vs mesurar

Dues parts complementàries sobre el mateix sistema:

**Part A — Camp estàtic (Blume-Capel).** Del model multinomial extreiem el camp $h$ i l'anisotropia $D$
i descomponem com hi contribueixen temperatura, sol i vent. Això és **inferència** des de l'ocupació.

**Part B — Transicions reals.** Com que tenim el mateix participant en parades consecutives, **mesurem
directament** cap a on flueix el sistema (onset, recovery, comfort-loss) en funció de T_corr i T_env, i
comprovem si la direcció coincideix amb el camp i, sobretot, **quin és el mecanisme** (que el camp no pot dir).

> Tesi de fons: l'ocupació estàtica diu *on* seu el sistema; les transicions diuen *com* hi arriba.
> El camp $h$ és una funció de l'ocupació — les transicions el contenen com a cas particular i n'afegeixen la cinètica.


In [ ]:
import pandas as pd, numpy as np, warnings, matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
warnings.filterwarnings('ignore'); np.random.seed(0)
plt.rcParams.update({'figure.dpi':120,'font.size':10})

CSV='../../data/markovian_analysis_baseline.csv'   # fitxer de transicions (vot-individual amb ID, stop_idx)
K_SUN, K_WIND = 8.5, 14.0               # coeficients T_env calibrats
SUN_MAP={'In full sun':1.0,'In a mixture of sun and shadow':0.5,'In full shade':0.0}
WIND_MAP={"It's not windy":0.0,"A very light wind":0.25,"A light wind":0.5,
          "A moderate wind":0.75,"A strong wind":1.0}

In [ ]:
# ---------- LOAD + COORDINATES ----------
df=pd.read_csv(CSV)
df=df.dropna(subset=['ID','stop_idx','walk_id','comfort3','<T-T_fixed+<T>>','sun','wind']).copy()
df['sun_s']=df['sun'].map(SUN_MAP); df['wind_s']=df['wind'].map(WIND_MAP)
df['T_corr']=df['<T-T_fixed+<T>>']
df['T_env']=df['T_corr']+K_SUN*df['sun_s']-K_WIND*df['wind_s']
df=df.dropna(subset=['sun_s','wind_s']).reset_index(drop=True)
print(f"vots={len(df)}  participants={df['ID'].nunique()}  caminades={df['walk_id'].nunique()}")
print("estats:",df['comfort3'].value_counts().to_dict())

## Part A — Descomposició del camp $h$ i l'anisotropia $D$

$E(S)=-hS+DS^2$ amb S = −1 (comfort), 0 (neutre), +1 (discomfort). Del multinomial (N base):
$h=\tfrac12\log\frac{P(U)}{P(C)}$, $D=-\tfrac12\log\frac{P(U)P(C)}{P(N)^2}$.
Equivalentment, amb $b_U=$ coef. de logit(U/N) i $b_C=$ coef. de logit(C/N):
$\partial h=\tfrac12(b_U-b_C)$, $\partial D=-\tfrac12(b_U+b_C)$.


In [ ]:
# ---------- FIELD DECOMPOSITION (individual level) ----------
v=df[['walk_id','comfort3','T_corr','sun_s','wind_s']].copy()
v.columns=['walk','side3','T','sun','wind']
v['side']=v['side3'].map({'comfortable':'C','neutral':'N','uncomfortable':'U'})
for c in ['T','sun','wind']:
    v[c+'_z']=(v[c]-v[c].mean())/v[c].std()
preds=['T_z','sun_z','wind_z']; names=['T','sol','vent']

def fit_field(data):
    yc=pd.Categorical(data['side'],categories=['N','C','U'])
    X=sm.add_constant(data[preds].to_numpy())
    m=sm.MNLogit(yc.codes,X).fit(disp=0,maxiter=200)
    return m.params[1:,0], m.params[1:,1]   # bC, bU (skip const)

bC0,bU0=fit_field(v); h0=0.5*(bU0-bC0); D0=-0.5*(bU0+bC0)
walks=v['walk'].unique(); BBU=[];BBC=[]
for _ in range(400):
    b=pd.concat([v[v['walk']==w] for w in np.random.choice(walks,len(walks),True)])
    if b['side'].nunique()<3: continue
    try: bc,bu=fit_field(b); BBU.append(bu);BBC.append(bc)
    except: pass
BBU,BBC=np.array(BBU),np.array(BBC); BH=0.5*(BBU-BBC); BD=-0.5*(BBU+BBC)

fig,axes=plt.subplots(1,4,figsize=(14,4)); fig.subplots_adjust(wspace=.35)
colors=['#185FA5','#D97B00','#1D9E75']
def panel(ax,vals,boot,title,xlabel):
    y=[2,1,0]
    for yi,(vv,bo,c) in enumerate(zip(vals,boot,colors)):
        lo,hi=np.percentile(bo,[2.5,97.5]); sig='*' if (lo>0)==(hi>0) else ''
        ax.barh(y[yi],vv,.35,color=c,alpha=.85)
        ax.errorbar(vv,y[yi],xerr=[[vv-lo],[hi-vv]],fmt='none',color='k',capsize=4,lw=1.4)
        ax.text(hi+.01,y[yi],sig,va='center',fontsize=14)
    ax.axvline(0,color='k',ls='--',lw=1,alpha=.7); ax.set_yticks(y); ax.set_yticklabels(names)
    ax.set_xlabel(xlabel); ax.set_title(title,fontsize=10); ax.grid(axis='x',alpha=.3)
panel(axes[0],bU0,[BBU[:,i] for i in range(3)],'bU (discomfort vs N)','log-odds/SD')
panel(axes[1],bC0,[BBC[:,i] for i in range(3)],'bC (comfort vs N)','log-odds/SD')
panel(axes[2],h0,[BH[:,i] for i in range(3)],'camp h = ½(bU−bC)','∂h/SD')
panel(axes[3],D0,[BD[:,i] for i in range(3)],'anisotropia D = −½(bU+bC)','∂D/SD')
fig.text(0.5,1.02,'bU significatiu i ~igual per T/sol/vent;  bC ≈ 0 (no significatiu)  →  '
         'l\u2019entorn expulsa del neutre cap a discomfort, no cap a comfort',
         ha='center',fontsize=10,bbox=dict(boxstyle='round',fc='#FFF3CD',ec='#D97B00'))
plt.savefig('fig_field.png',bbox_inches='tight'); plt.show()
print("bU:",dict(zip(names,np.round(bU0,3))),"  bC:",dict(zip(names,np.round(bC0,3))))

## Part B — Transicions reals (mesurar, no inferir)

Construïm transicions individuals: mateix `ID`, `stop_idx` consecutiu dins de la mateixa `walk_id`.
Condicionem per les condicions a la parada **destí** t+1 (la resposta nova es dóna sota les condicions noves).
Alternatives defensables: origen t, o canvi $\Delta T$ — canvieu `COND` per explorar-les.


In [ ]:
# ---------- BUILD INDIVIDUAL TRANSITIONS ----------
df=df.sort_values(['ID','walk_id','stop_idx'])
tr=[]
for (pid,w),g in df.groupby(['ID','walk_id']):
    rec=g.sort_values('stop_idx').to_dict('records')
    for a,b in zip(rec[:-1],rec[1:]):
        if b['stop_idx']-a['stop_idx']==1:
            tr.append(dict(s0=a['comfort3'],s1=b['comfort3'],
                           T0=a['T_corr'],T1=b['T_corr'],Te0=a['T_env'],Te1=b['T_env']))
T=pd.DataFrame(tr)
print(f"transicions individuals consecutives: {len(T)}")
print("\nMatriu de transicio agregada (comptes):")
print(pd.crosstab(T.s0,T.s1).reindex(index=['comfortable','neutral','uncomfortable'],
                                      columns=['comfortable','neutral','uncomfortable']))

In [ ]:
# ---------- FLOWS vs TEMPERATURE ----------
def wilson(k,n):
    if n==0: return (np.nan,np.nan,np.nan)
    z=1.96;p=k/n;den=1+z*z/n;c=(p+z*z/(2*n))/den
    h=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/den; return p,c-h,c+h

def flows(col,nb=4):
    T['b']=pd.qcut(T[col],nb,duplicates='drop')
    out=[]
    for b,g in T.groupby('b',observed=True):
        gN=g[g.s0=='neutral']; gU=g[g.s0=='uncomfortable']; gC=g[g.s0=='comfortable']
        out.append(dict(t=g[col].mean(),
            onset=wilson((gN.s1=='uncomfortable').sum(),len(gN)),
            recov=wilson((gU.s1!='uncomfortable').sum(),len(gU)),
            closs=wilson((gC.s1!='comfortable').sum(),len(gC))))
    # transition-level spearman
    gN=T[T.s0=='neutral']; gU=T[T.s0=='uncomfortable']; gC=T[T.s0=='comfortable']
    rho=dict(onset=stats.spearmanr(gN[col],(gN.s1=='uncomfortable')).statistic,
             recov=stats.spearmanr(gU[col],(gU.s1!='uncomfortable')).statistic,
             closs=stats.spearmanr(gC[col],(gC.s1!='comfortable')).statistic)
    return pd.DataFrame(out), rho

fig,axes=plt.subplots(1,2,figsize=(13,4.6),sharey=True)
for ax,col in zip(axes,['T1','Te1']):
    d,rho=flows(col)
    for key,c,lab in [('onset','#D85A30','onset N→U'),('recov','#185FA5','recovery U→·'),
                      ('closs','#1D9E75','comfort-loss C→·')]:
        x=d['t']; y=[v[0] for v in d[key]]; lo=[v[1] for v in d[key]]; hi=[v[2] for v in d[key]]
        ax.plot(x,y,'-o',color=c,label=f"{lab}  (ρ={rho[key]:+.2f})")
        ax.fill_between(x,lo,hi,color=c,alpha=.13)
    ax.axvline(29,ls='--',color='gray',lw=1)
    ax.set_xlabel('T_corr (°C)' if col=='T1' else 'T_env (°C)')
    ax.set_title(('T_corr' if col=='T1' else 'T_env')+' (condicions destí)')
    ax.grid(alpha=.3); ax.legend(fontsize=8.5)
axes[0].set_ylabel('probabilitat de transició')
plt.suptitle('Fluxos de transició vs temperatura — mesurats, no inferits',y=1.02)
plt.savefig('fig_flows.png',bbox_inches='tight'); plt.show()

## Síntesi: el camp coincideix amb el flux?

- **Direcció:** el camp $h$ deia que la calor empeny cap a discomfort. Els fluxos ho confirmen
  (onset puja, recovery baixa amb T). ✓
- **Mecanisme (el que el camp NO pot dir):** comparant les pendents ρ, la recovery respon molt més
  fortament a la temperatura que l'onset → el sistema és **dominat per trapping**, no per onset.
- **T_env vs T_corr:** la recovery és molt més monòtona i amb |ρ| més gran sota T_env → la dinàmica
  confirma T_env com a millor coordenada, des d'evidència independent de la separació estàtica.
- **Comfort no és inert:** el comfort-loss C→· és fortament dependent de T_env, tot i que bC≈0 a
  l'anàlisi estàtica. L'ocupació amagava el flux que la dinàmica revela.


In [ ]:
# ---------- QUANTITATIVE SYNTHESIS ----------
_,rho_corr=flows('T1'); _,rho_env=flows('Te1')
print("Pendent (Spearman) flux vs temperatura:")
print(f"{'':14}{'T_corr':>10}{'T_env':>10}")
for k,lab in [('onset','onset N→U'),('recov','recovery U→·'),('closs','comfort-loss C→·')]:
    print(f"{lab:14}{rho_corr[k]:>10.3f}{rho_env[k]:>10.3f}")
print("\nMecanisme: |ρ(recovery)| >> |ρ(onset)|  =>  TRAPPING-dominat")
print("Coordenada: |ρ| mes gran amb T_env  =>  T_env organitza millor la dinamica")
summary=pd.DataFrame({'T_corr':rho_corr,'T_env':rho_env})
summary.to_csv('flows_vs_T_summary.csv')